In [1]:
import json
from cap.paths import get_paths

paths = get_paths()
stages = ["00_cohort", "01_preprocessing", "02_cell_annotation",
          "03a_epithelial_integration", "03b_meta_programmes",
          "03c_antigen_coexpression", "03d_normal_tissue_protein",
          "04_programme_summary", "05_three_proteome"]

for s in stages:
    d = paths.stage_dir(s)
    f = d / "run_provenance.json"
    if not f.is_file():
        print(f"{s:30} NO PROVENANCE"); continue
    p = json.loads(f.read_text(encoding="utf-8"))
    nb = (p.get("notebook") or {}).get("path")
    print(f"{s:30} run={p['run_id'] == paths.run_id} dirty={p['git_dirty']} "
          f"complete={p['provenance_completeness']['complete']} "
          f"commit={p['git_commit'][:8]} nb={nb}")

00_cohort                      run=True dirty=False complete=True commit=74d4395d nb=notebooks/000_Raw_Data_And_Cohort.ipynb
01_preprocessing               run=True dirty=True complete=True commit=9124e1a6 nb=notebooks/001_Papermill.ipynb
02_cell_annotation             run=True dirty=True complete=True commit=9124e1a6 nb=notebooks/002_Celltype_ID.ipynb
03a_epithelial_integration     run=True dirty=True complete=False commit=9124e1a6 nb=None
03b_meta_programmes            run=True dirty=True complete=False commit=9124e1a6 nb=None
03c_antigen_coexpression       run=True dirty=True complete=False commit=9124e1a6 nb=None
03d_normal_tissue_protein      run=True dirty=True complete=True commit=9124e1a6 nb=notebooks/003d_Normal_Tissue_Protein.ipynb
04_programme_summary           run=True dirty=True complete=False commit=9124e1a6 nb=None
05_three_proteome              run=True dirty=True complete=False commit=902e2f9b nb=None


In [2]:
import json
from collections import Counter
from cap.paths import get_paths
paths = get_paths()

for s in ["01_preprocessing", "02_cell_annotation", "03a_epithelial_integration",
          "03b_meta_programmes", "03c_antigen_coexpression",
          "03d_normal_tissue_protein", "04_programme_summary", "05_three_proteome"]:
    p = json.loads((paths.stage_dir(s) / "run_provenance.json").read_text(encoding="utf-8"))
    lines = p["git_status_porcelain"]
    print(f"\n{s}: {len(lines)} entries")
    pycharm = [l for l in lines if "pycharm-support-libs" in l]
    other = [l for l in lines if "pycharm-support-libs" not in l]
    print("  pycharm noise:", len(pycharm))
    for l in other[:15]:
        print("  ", l)


01_preprocessing: 1 entries
  pycharm noise: 0
    M notebooks/001_Papermill.ipynb

02_cell_annotation: 1 entries
  pycharm noise: 0
    M notebooks/001_Papermill.ipynb

03a_epithelial_integration: 2 entries
  pycharm noise: 0
    M notebooks/001_Papermill.ipynb
    M notebooks/002_Celltype_ID.ipynb

03b_meta_programmes: 4 entries
  pycharm noise: 0
    M notebooks/001_Papermill.ipynb
    M notebooks/002_Celltype_ID.ipynb
    M notebooks/003a_Epithelial_Integration.ipynb
    M notebooks/003b_Meta_Programmes.ipynb

03c_antigen_coexpression: 5 entries
  pycharm noise: 0
    M notebooks/001_Papermill.ipynb
    M notebooks/002_Celltype_ID.ipynb
    M notebooks/003a_Epithelial_Integration.ipynb
    M notebooks/003b_Meta_Programmes.ipynb
    M notebooks/003c_Antigen_Coexpression.ipynb

03d_normal_tissue_protein: 5 entries
  pycharm noise: 0
    M notebooks/001_Papermill.ipynb
    M notebooks/002_Celltype_ID.ipynb
    M notebooks/003a_Epithelial_Integration.ipynb
    M notebooks/003b_Meta_Pr

# Raw data and cohort definition

## Imports


In [1]:
from __future__ import annotations
import json
from pathlib import Path
import pandas as pd
import session_info


## Paths


In [2]:
from cap.paths import get_paths
from cap.provenance import sha256_file, write_run_provenance

paths = get_paths()

raw_dir = paths.raw_counts_dir
features_file = paths.features_file

paths.require_dir(raw_dir, "GSE161529 filtered count directory")
paths.require_file(features_file, "shared features.tsv")


WindowsPath('D:/CAP_Data/raw/gse161529/features.tsv')

The cohort is selected from the raw library identifiers using the rules below.

In [3]:
candidate_stage_dir = paths.stage_dir("00_cohort")

print(f"CAP_RUN_ID: {paths.run_id}")
print(f"CAP_DATA_ROOT: {paths.data_root}")
print(f"00_cohort stage: {candidate_stage_dir}")

if "replace_with" in paths.run_id.casefold():
    raise ValueError(
        "CAP_RUN_ID is still a template value. Set a unique CAP_RUN_ID in the "
        "environment or repository .env, then restart the kernel."
    )

if candidate_stage_dir.exists() and (
    not candidate_stage_dir.is_dir() or any(candidate_stage_dir.iterdir())
):
    raise FileExistsError(
        f"Stage output already exists and is non-empty: {candidate_stage_dir}. "
        f"The effective CAP_RUN_ID is {paths.run_id!r}. Set a fresh unique "
        "CAP_RUN_ID and restart the kernel; existing stage outputs are not overwritten."
    )

# Reserve/create the stage only after input validation is complete.


CAP_RUN_ID: preprocessing_test_008
CAP_DATA_ROOT: D:\CAP_Data
00_cohort stage: D:\CAP_Data\runs\preprocessing_test_008\00_cohort


In [4]:
def display_path(path: Path) -> Path:
    """Display paths without exposing local absolute directories."""
    path = Path(path).resolve()

    try:
        return Path("<CAP_DATA_ROOT>") / path.relative_to(paths.data_root)
    except ValueError:
        try:
            return Path("<REPOSITORY>") / path.relative_to(paths.root)
        except ValueError:
            return Path("<external>") / path.name


In [5]:
def sample_paths(sample_id):
    return (
        raw_dir / f"{sample_id}-matrix.mtx.gz",
        raw_dir / f"{sample_id}-barcodes.tsv.gz",
    )


In [6]:
matrices = sorted(raw_dir.glob("*-matrix.mtx.gz"))


In [7]:
barcodes = sorted(raw_dir.glob("*-barcodes.tsv.gz"))


In [8]:
assert matrices, f"no matrix files under {display_path(raw_dir)}"


In [9]:
assert len(matrices) == len(barcodes), (len(matrices), len(barcodes))

matrix_ids = {path.name.removesuffix("-matrix.mtx.gz") for path in matrices}
barcode_ids = {path.name.removesuffix("-barcodes.tsv.gz") for path in barcodes}
if matrix_ids != barcode_ids:
    raise ValueError(
        "Matrix and barcode sample IDs do not match. "
        f"Missing barcodes: {sorted(matrix_ids - barcode_ids)}; "
        f"missing matrices: {sorted(barcode_ids - matrix_ids)}"
    )


In [10]:
print(f"raw counts directory : {display_path(raw_dir)}")


raw counts directory : <CAP_DATA_ROOT>\raw\GSE161529\raw_counts


In [11]:
print(f"matrix files         : {len(matrices)}")


matrix files         : 69


In [12]:
print(f"barcode files        : {len(barcodes)}")


barcode files        : 69


In [13]:
print(f"shared features.tsv  : {display_path(features_file)}")


shared features.tsv  : <CAP_DATA_ROOT>\raw\GSE161529\features.tsv


In [14]:
accessions = sorted({p.name.split("_")[0][:7] for p in matrices})


In [15]:
print("accession prefixes present:", accessions)


accession prefixes present: ['GSM4909']


In [16]:
example = matrices[0].name


In [17]:
print(f"\nexample filename: {example}")



example filename: GSM4909253_N-PM0092-Total-matrix.mtx.gz


In [18]:
def parse_sample_name(stem):
    """Turn 'GSM4909285_TN-B1-MH4031' into structured metadata."""
    conditions = {"N", "TN", "HER2", "ER", "mER", "B1"}
    sort_labels = {"Total", "Epi"}
    site_labels = {"LN"}
    gsm, _, rest = stem.partition("_")
    tokens = rest.split("-")
    condition = tokens[0]
    if condition not in conditions:
        raise ValueError(f"unrecognised condition {condition!r} in {stem!r}")
    idx, brca1 = (1, condition == "B1")
    if len(tokens) > 1 and tokens[1] == "B1":
        brca1, idx = (True, 2)
    patient = tokens[idx]
    qualifiers = tokens[idx + 1 :]
    return {
        "sample_id": stem,
        "gsm": gsm,
        "condition": condition,
        "patient": patient,
        "brca1_carrier": brca1,
        "sorting": next((q for q in qualifiers if q in sort_labels), "unsorted"),
        "site": (
            "lymph_node" if any((q in site_labels for q in qualifiers)) else "primary"
        ),
        "replicate": next(
            (q for q in qualifiers if q not in sort_labels and q not in site_labels), ""
        ),
    }


In [19]:
demo = "GSM4909285_TN-B1-MH4031"


In [20]:
for key, value in parse_sample_name(demo).items():
    print(f"  {key:16s} {value}")


  sample_id        GSM4909285_TN-B1-MH4031
  gsm              GSM4909285
  condition        TN
  patient          MH4031
  brca1_carrier    True
  sorting          unsorted
  site             primary
  replicate        


In [21]:
samples = pd.DataFrame(
    [parse_sample_name(p.name.replace("-matrix.mtx.gz", "")) for p in matrices]
)


In [22]:
print(f"{len(samples)} samples\n")


69 samples



In [23]:
print(pd.crosstab(samples["condition"], samples["sorting"]).to_string())


sorting    Epi  Total  unsorted
condition                      
B1           0      0         4
ER           0      0        24
HER2         0      0         6
N           11     13         0
TN           0      0         8
mER          0      0         3


In [24]:
print(pd.crosstab(samples["condition"], samples["site"]).to_string())


site       lymph_node  primary
condition                     
B1                  0        4
ER                  6       18
HER2                0        6
N                   0       24
TN                  0        8
mER                 1        2


In [25]:
def decide_inclusion(row, has_total):
    """rules are applied in order, so a sample is counted under the first that matches"""
    if row["condition"] == "mER":
        return (
            False,
            "male breast cancer; distinct biology and sex chromosome complement",
        )
    if row["condition"] == "B1":
        return (False, "BRCA1 preneoplastic tissue; neither normal nor tumour")
    if row["site"] == "lymph_node":
        return (False, "lymph node metastasis; cohort restricted to primary tissue")
    if row["sorting"] == "Epi":
        if row["patient"] in has_total:
            return (
                False,
                "epithelially sorted; unsorted -Total sample used for this patient",
            )
        return (
            False,
            "epithelially sorted; cell-type composition not comparable with the unsorted -Total samples in the rest of the cohort",
        )
    return (True, "")


In [26]:
has_total = set(samples.loc[samples["sorting"] == "Total", "patient"])


In [27]:
samples["include"], samples["exclusion_reason"] = zip(
    *samples.apply(lambda row: decide_inclusion(row, has_total), axis=1)
)


Both MH0114 libraries are excluded because their deposited subtype annotations conflict and their clinical relationship remains unresolved.

In [28]:
excluded_patients = {
    "MH0114": (
        "Different deposited subtype annotations for specimens from the same donor; "
        "their clinical relationship could not be resolved from available metadata"
    )
}


In [29]:
samples["include_before_patient_exclusion"] = samples["include"]


In [30]:
patient_exclusion = samples["patient"].isin(excluded_patients)


In [31]:
samples.loc[patient_exclusion, "include"] = False


In [32]:
samples.loc[patient_exclusion, "exclusion_reason"] = samples.loc[
    patient_exclusion, "patient"
].map(excluded_patients)


In [33]:
included = samples[samples["include"]]


In [34]:
print(f"{len(samples)} samples found -> {len(included)} included\n")


69 samples found -> 43 included



In [35]:
print("included:")


included:


In [36]:
print(
    included.groupby("condition")
    .agg(
        n_samples=("sample_id", "size"),
        n_patients=("patient", "nunique"),
        n_brca1=("brca1_carrier", "sum"),
    )
    .to_string()
)


           n_samples  n_patients  n_brca1
condition                                
ER                17          16        0
HER2               6           6        0
N                 13          13        0
TN                 7           7        4


In [37]:
print("\nexcluded:")



excluded:


In [38]:
for reason, n in (
    samples.loc[~samples["include"], "exclusion_reason"].value_counts().items()
):
    print(f"  {n:3d}  {reason}")


    6  epithelially sorted; unsorted -Total sample used for this patient
    6  lymph node metastasis; cohort restricted to primary tissue
    5  epithelially sorted; cell-type composition not comparable with the unsorted -Total samples in the rest of the cohort
    4  BRCA1 preneoplastic tissue; neither normal nor tumour
    3  male breast cancer; distinct biology and sex chromosome complement
    2  Different deposited subtype annotations for specimens from the same donor; their clinical relationship could not be resolved from available metadata


In [39]:
repeated = included[included.duplicated("patient", keep=False)].sort_values(
    ["patient", "sample_id"]
)


In [40]:
print(
    f"{len(included)} samples from {included['patient'].nunique()} distinct patients\n"
)


43 samples from 41 distinct patients



In [41]:
for patient, group in repeated.groupby("patient"):
    conditions = sorted(set(group["condition"]))
    flag = "    -> spans conditions" if len(conditions) > 1 else ""
    print(f"  {patient:10s} {list(group['sample_id'])}{flag}")


  MH0029     ['GSM4909305_ER-MH0029-7C', 'GSM4909306_ER-MH0029-9C']
  MH0064     ['GSM4909263_N-MH0064-Total', 'GSM4909313_ER-MH0064-T']    -> spans conditions


##  Write the sample sheet

In [42]:
def files_present(sample_id):
    matrix, barcode = sample_paths(sample_id)
    return matrix.exists() and barcode.exists()


In [43]:
samples["files_present"] = samples["sample_id"].map(files_present)


In [44]:
missing = samples.loc[samples["include"] & ~samples["files_present"], "sample_id"]


In [45]:
assert missing.empty, f"missing raw files for: {list(missing)}"


In [46]:
samples = samples.sort_values(["condition", "patient", "sample_id"]).reset_index(
    drop=True
)


In [47]:
stage_dir = paths.prepare_stage("00_cohort")

df_dir = stage_dir / "tables"
df_dir.mkdir(parents=True, exist_ok=True)
sample_sheet = df_dir / "cohort_samples.csv"


In [48]:
samples.to_csv(sample_sheet, index=False, lineterminator="\n")

In [49]:
assert len(samples) == 69, len(samples)


In [50]:
assert samples["include"].sum() == (43 if excluded_patients else 45)


In [51]:
assert included["patient"].nunique() == (41 if excluded_patients else 42)


In [52]:
print(f"written: {display_path(sample_sheet)}")


written: <CAP_DATA_ROOT>\runs\preprocessing_test_008\00_cohort\tables\cohort_samples.csv


In [53]:
print(f"  {len(samples)} rows, {samples['include'].sum()} included")


  69 rows, 43 included


In [54]:
print(f"  columns: {', '.join(samples.columns)}")


  columns: sample_id, gsm, condition, patient, brca1_carrier, sorting, site, replicate, include, exclusion_reason, include_before_patient_exclusion, files_present


In [55]:
size_mb = {
    sample_id: sample_paths(sample_id)[0].stat().st_size / 1000000.0
    for sample_id in included["sample_id"]
}


In [56]:
sizes = (
    included[["sample_id", "condition"]]
    .assign(size_mb=lambda d: d["sample_id"].map(size_mb))
    .sort_values("size_mb")
)


In [57]:
print("smallest samples, good candidates for demo_id:\n")


smallest samples, good candidates for demo_id:



In [58]:
print(sizes.head(5).to_string(index=False, float_format=lambda v: f"{v:.1f}"))


                sample_id condition  size_mb
   GSM4909292_HER2-MH0069      HER2      2.4
   GSM4909311_ER-MH0056-T        ER      6.4
     GSM4909283_TN-SH0106        TN      6.7
     GSM4909300_ER-MH0032        ER      7.0
GSM4909272_N-MH0021-Total         N      7.2


In [59]:
print(f'''\nsuggested:\n    demo_id = "{sizes.iloc[0]['sample_id']}"''')



suggested:
    demo_id = "GSM4909292_HER2-MH0069"


In [60]:
input_files = {"features.tsv": features_file}
for input_path in [*matrices, *barcodes]:
    input_files[f"raw_counts/{input_path.name}"] = input_path

input_checksums = {
    name: sha256_file(path)
    for name, path in sorted(input_files.items())
}

checksums_file = stage_dir / "input_checksums.json"
checksums_file.write_text(
    json.dumps(input_checksums, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

provenance_file = write_run_provenance(
    paths,
    stage_dir,
    notebook_path=paths.root / "notebooks" / "000_Raw_Data_And_Cohort.ipynb",
    input_checksums=input_checksums,
)
print(f"Input checksums: {len(input_checksums)} files")
print(f"Run provenance: {display_path(provenance_file)}")


Input checksums: 139 files
Run provenance: <CAP_DATA_ROOT>\runs\preprocessing_test_008\00_cohort\run_provenance.json


## Per-library preprocessing

This notebook only defines the cohort. Run preprocessing with the dedicated `001_Papermill` driver using the same `CAP_RUN_ID`.


In [61]:
session_info.show()


In [62]:
import json

prov_file = stage_dir / "run_provenance.json"
checksums_file = stage_dir / "input_checksums.json"

prov = json.loads(prov_file.read_text(encoding="utf-8"))
checksums = json.loads(checksums_file.read_text(encoding="utf-8"))

print("run ID:", prov["run_id"])
print("Git dirty:", prov["git_dirty"])
print("Provenance completeness:", prov["provenance_completeness"])
print("Checksum entries:", len(checksums))
print("Cohort table exists:", (stage_dir / "tables" / "cohort_samples.csv").is_file())

run ID: preprocessing_test_008
Git dirty: True
Provenance completeness: {'complete': True, 'missing_fields': []}
Checksum entries: 139
Cohort table exists: True
